## Downscaling with the ViT model

This notebook showcases a simple application of deep4downscaling for the statistical downscaling of precipitation using the Vision Transformer (ViT) architecture [1]. In particular, we use the NoisyViT variant [2], which injects noise into the transformer encoder to generate stochastic outputs, enabling training with the CRPS loss function. To do so, we will implement the following actions:

- Define and train the NoisyViT architecture.
- Downscale and evaluate results over a test period.

### Train the model

In [ ]:
DATA_PATH = './data/input'
FIGURES_PATH = './figures'
MODELS_PATH = './models'

When working with climate data, xarray is an essential library, and deep4downscaling heavily relies on it. For the deep learning component, deep4downscaling uses PyTorch, one of the most popular frameworks in the field.

In [ ]:
import xarray as xr
import torch
from torch.utils.data import DataLoader, random_split
import numpy as np

import deep4downscaling.viz
import deep4downscaling.trans
import deep4downscaling.deep.loss
import deep4downscaling.deep.utils
import deep4downscaling.deep.models
import deep4downscaling.deep.train
import deep4downscaling.deep.pred

We will begin by loading the predictor. In this case, we select various large-scale variables from ERA5 at different height levels. These variables are already stored in a NetCDF file, the standard data format for deep4downscaling. Unfortunately, due to GitHub's size restrictions, we are unable to upload these files to the repository. However, the following cells provide an overview of the data, making it straightforward to reproduce this notebook with a similar file.

In [ ]:
predictor_filename = f'{DATA_PATH}/ERA5_NorthAtlanticRegion_1-5dg_full.nc'
predictor = xr.open_dataset(predictor_filename)

In [ ]:
predictor

The deep4downscaling library provides several functions to facilitate an initial visualization of the data. For example, the `deep4downscaling.viz.multiple_map_plot` function allows you to visualize all the variables in the dataset.

In [ ]:
deep4downscaling.viz.multiple_map_plot(data=predictor.mean('time'),
                                       output_path=f'./{FIGURES_PATH}/predictor_climatology.pdf')

#### Adjusting predictor dimensions for the ViT

The ViT model splits the input into fixed-size, non-overlapping patches for building token embeddings. This imposes constraints on the spatial dimensions of the predictor:

- Both the height and width of the input must be **divisible by the patch size**.
- The spatial resolution of the output (predictand) must be a **multiple of the number of input tokens** per spatial dimension (i.e., `height / patch_size`).

In practice, this means the predictor grid may need to be extended to meet these constraints. Here, we extend the latitude dimension to 32 grid points by reindexing with the nearest available value.

In [ ]:
lat = predictor.lat.values
dlat = np.diff(lat).mean()
n_extra = 32 - lat.size
new_lat = np.concatenate([lat, lat[-1] + dlat * np.arange(1, n_extra + 1)])
predictor = predictor.reindex(lat=new_lat, method='nearest')

The predictand is an `xarray.Dataset` containing a single variable (the target). In this notebook, we will focus on downscaling accumulated precipitation over the region of peninsular Spain and the Balearic Islands.

In [ ]:
predictand_filename = f'{DATA_PATH}/pr_AEMET.nc'
predictand = xr.open_dataset(predictand_filename)

In [ ]:
predictand

Similar to the predictors, deep4downscaling can also be used for an initial visualization of the predictand.

In [ ]:
day_to_viz = '10-04-2015'
deep4downscaling.viz.simple_map_plot(data=predictand.sel(time=day_to_viz),
                                     colorbar='hot_r', var_to_plot='pr',
                                     output_path=f'./{FIGURES_PATH}/predictand_day.pdf')

#### Adjusting predictand dimensions for the ViT

Similarly to the predictor, the predictand grid must also be adjusted. The ViT model internally computes the output spatial dimensions as `H_out = W_out = sqrt(num_outputs)`, meaning the **total number of output grid points must be a perfect square** (i.e., the output grid must be square). Additionally, `H_out` must be divisible by the number of input tokens per dimension.

Here, we extend the latitude dimension to 256 grid points. Note that in this example the longitude dimension already has the right number of grid points.

In [ ]:
lat = predictand.lat.values
dlat = np.diff(lat).mean()
n_extra = 256 - lat.size
new_lat = np.concatenate([lat, lat[-1] + dlat * np.arange(1, n_extra + 1)])
predictand = predictand.reindex(lat=new_lat)

Deep4downscaling also includes several common preprocessing techniques used in statistical downscaling, such as removing NaN values, aligning datasets (e.g., across time), bias adjustment, and standardization.

In [ ]:
predictor = deep4downscaling.trans.remove_days_with_nans(predictor)

predictor, predictand = deep4downscaling.trans.align_datasets(predictor, predictand, 'time')

To adhere to the standard training/validation scheme in the machine learning field, we divide the predictors and predictand into training and test sets.

In [ ]:
years_train = ('1980', '2010')
years_test = ('2011', '2020')

x_train = predictor.sel(time=slice(*years_train))
y_train = predictand.sel(time=slice(*years_train))

x_test = predictor.sel(time=slice(*years_test))
y_test = predictand.sel(time=slice(*years_test))

Before feeding the predictors to the deep learning model, we standardize them to have a mean of zero and a standard deviation of one. This is done using the `deep4downscaling.trans.standardize` function, which stores the training statistics to later standardize the test set consistently.

In [ ]:
x_train_stand = deep4downscaling.trans.standardize(data_ref=x_train, data=x_train)

The mask identifies valid (non-NaN) grid points and is required during the prediction phase to convert the raw model output back into a properly formatted `xr.Dataset` with the correct spatial structure.

In [ ]:
y_mask = deep4downscaling.trans.compute_valid_mask(y_train)

All deep learning models implemented in deep4downscaling flatten their output into a vector, standardizing its dimensions to the shape `(time, grid point)`.

Note that, unlike CNN-based models such as DeepESD, the ViT requires the full spatial grid to be preserved (including NaN grid points). This is because the model internally reconstructs a square spatial output from patch-level predictions, so filtering out grid points would break the spatial structure.

In [ ]:
y_train_stack = y_train.stack(gridpoint=('lat', 'lon'))

The deep4downscaling library includes various loss functions for training deep learning models. In this notebook, we use the CRPS (Continuous Ranked Probability Score) loss function [2]. The CRPS loss requires the model to produce multiple stochastic outputs (ensemble members), which is why we use the NoisyViT architecture. The `ignore_nans` flag ensures that NaN grid points in the target are properly handled during training.

In [ ]:
loss_function = deep4downscaling.deep.loss.CRPSLoss(ignore_nans=True)

NetCDF is not well-suited for use with PyTorch (or for converting to the `torch.Tensor` type). In contrast, NumPy is.

In [ ]:
x_train_stand_arr = deep4downscaling.trans.xarray_to_numpy(x_train_stand)
y_train_arr = deep4downscaling.trans.xarray_to_numpy(y_train_stack)

With our data now in the numpy format, we can create the `torch.Dataset` and `torch.DataLoader` to feed batches of data to the deep learning model during training.

In [ ]:
train_dataset = deep4downscaling.deep.utils.StandardDataset(x=x_train_stand_arr,
                                                            y=y_train_arr)

train_dataset, valid_dataset = random_split(train_dataset,
                                            [0.9, 0.1])

batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size,
                              shuffle=True)

Deep4downscaling includes several predefined deep learning architectures (e.g., DeepESD and U-Net), but custom architectures can be easily defined using the standard PyTorch framework. However, because deep4downscaling relies on a final flattening operation (as mentioned earlier), we recommend reviewing the implementations in `deep4downscaling.deep.models` and using them as a foundation.

While deep4downscaling lacks a formal documentation page, all its functions and arguments are properly documented within the code.

In [ ]:
?deep4downscaling.deep.models.NoisyViT

#### NoisyViT model parameters

In this notebook, we will train the NoisyViT architecture for statistical downscaling. The NoisyViT extends the base ViT [1] by injecting noise into the transformer encoder, enabling stochastic outputs as proposed in [2]. During training, the model generates multiple ensemble members per input, which are used to compute the CRPS loss. During inference, each forward pass produces a single stochastic realization.

The NoisyViT shares all the base ViT parameters and adds noise-specific ones:

| Parameter | Type | Description |
|-----------|------|-------------|
| `x_shape` | tuple | Shape of the input data `(batch, channels, height, width)`. Both height and width must be divisible by `patch_size`. |
| `y_shape` | tuple | Shape of the output data `(batch, num_outputs)`. `num_outputs` must be a perfect square (since the model assumes a square output grid). |
| `patch_size` | int | Size of the non-overlapping patches extracted from the input. Must divide both the height and width of the input. |
| `dim` | int | Dimension of the token embeddings. Must be divisible by `num_heads`. |
| `depth` | int | Number of transformer encoder blocks. |
| `num_heads` | int | Number of attention heads in each transformer block. |
| `mlp_dim` | int | Hidden dimension of the MLP within each transformer block. |
| `noise_channels` | int | Number of noise channels injected into the encoder. Controls the amount of stochasticity. |
| `noise_dim` | int | Dimension of the noise embeddings passed to the conditional layer normalization in each transformer block. |
| `members_for_training` | int | Number of ensemble members generated per input during training (default: 2). More members improve the CRPS estimate but increase memory usage. |
| `dropout` | float | Dropout probability (default: 0.0). |
| `orog` | torch.Tensor | Optional orography tensor of shape `(H_out, W_out)`. If provided, orography patches are embedded and added to the token representations before decoding. |
| `overlap` | int | Overlap between decoded patches (default: 0). When > 0, uses a Hann window and overlap-add reconstruction to reduce artifacts at patch boundaries, which is especially useful when noise is injected independently per patch. |
| `last_relu` | bool | If True, applies a ReLU activation to the final output (default: False). Useful for variables that must be non-negative (e.g., precipitation). |

#### Data format requirements and limitations

The ViT imposes specific constraints on the input and output data:

- **Input**: must be 4D with shape `(batch, channels, height, width)`. Both `height` and `width` must be divisible by `patch_size`.
- **Output**: must be 2D with shape `(batch, num_outputs)`. The model computes `H_out = W_out = sqrt(num_outputs)`, so `num_outputs` must be a **perfect square** (i.e., the output grid must be square).
- The output spatial resolution (`H_out`) must be divisible by the number of tokens per dimension (`height / patch_size`), since the model upscales each token to a local patch of the output grid.
- Unlike CNN-based models (e.g., DeepESD), **NaN grid points cannot be filtered out** before training, since the ViT reconstructs a full square spatial grid from its token-level predictions. Instead, the loss function handles NaN values via the `ignore_nans` flag.

In [ ]:
model_name = 'NoisyViT_pr'
model = deep4downscaling.deep.models.NoisyViT(x_shape=x_train_stand_arr.shape,
                                               y_shape=y_train_arr.shape,
                                               patch_size=2,
                                               dim=768,
                                               depth=12,
                                               num_heads=12,
                                               mlp_dim=3072,
                                               noise_channels=8,
                                               noise_dim=64,
                                               members_for_training=2,
                                               orog=None,
                                               last_relu=True)

We set the typical training hyperparameters, as is commonly done in PyTorch.

In [ ]:
num_epochs = 10000
patience_early_stopping = 20

learning_rate = 0.0001
optimizer = torch.optim.Adam(model.parameters(),
                             lr=learning_rate)

Deep learning models can run on either CPU or GPU devices. We provide the corresponding `.yml` environment files (`deep4downscaling/requirement`) to set up a basic Conda environment for running deep4downscaling on both CPU and GPU.

In [ ]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

Deep4downscaling provides the `deep4downscaling.deep.train.standard_training_loop`, which implements a basic training routine. Models are saved based on their performance on a validation set through an early stopping mechanism.

In [ ]:
train_loss, val_loss = deep4downscaling.deep.train.standard_training_loop(
                            model=model, model_name=model_name, model_path=MODELS_PATH,
                            device=device, num_epochs=num_epochs,
                            loss_function=loss_function, optimizer=optimizer,
                            train_data=train_dataloader, valid_data=valid_dataloader,
                            patience_early_stopping=patience_early_stopping)

### Downscale the test set

Once a model has been trained and saved as a `.pt` file, it is easy to compute predictions on a new set of predictors. In this example, we will compute predictions on the test set, which was subset a few cells above.

Since the NoisyViT is a stochastic model, we can generate an ensemble of predictions by setting the `ensemble_size` parameter. Each forward pass produces a different realization due to the injected noise.

In [ ]:
model.load_state_dict(torch.load(f'{MODELS_PATH}/{model_name}.pt'))

x_test_stand = deep4downscaling.trans.standardize(data_ref=x_train, data=x_test)

pred_test = deep4downscaling.deep.pred.compute_preds_standard(
                                x_data=x_test_stand, model=model,
                                device=device, var_target='pr',
                                mask=y_mask, ensemble_size=10,
                                batch_size=16)

In [ ]:
deep4downscaling.viz.simple_map_plot(data=pred_test.mean('time').mean('member'),
                                     colorbar='hot_r', var_to_plot='pr',
                                     output_path=f'./{FIGURES_PATH}/prediction_test_mean.pdf')

### References

[1] Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., ... & Houlsby, N. (2020). An image is worth 16x16 words: Transformers for image recognition at scale. arXiv preprint arXiv:2010.11929.

[2] Lang, S., Alexe, M., Clare, M. C., Roberts, C., Adewoyin, R., Bouallègue, Z. B., ... & Leutbecher, M. (2024). AIFS-CRPS: ensemble forecasting using a model trained with a loss function based on the continuous ranked probability score. arXiv preprint arXiv:2412.15832.